# L5: ML核心概念


# L5: ML核心概念

**课时**: 2 课时（90 分钟/课时）

## 1. 学习目标

完成本课程后，学员能够：
1. 区分监督学习、无监督学习、半监督学习与强化学习四类范式
2. 解释训练集、验证集、测试集的作用与划分原则
3. 理解偏差-方差权衡（Bias-Variance Tradeoff）的数学本质
4. 掌握过拟合、欠拟合的概念及正则化、Dropout 等缓解策略
5. 了解奥卡姆剃刀原则与VC维理论的基本思想

## 2. 核心知识点

### 2.1 机器学习四类范式

| 范式 | 特点 | 典型算法 | 典型应用 |
|------|------|----------|----------|
| 监督学习 | 有标签，映射学习 | 线性回归、决策树、神经网络 | 分类、回归 |
| 无监督学习 | 无标签，结构发现 | K-Means、PCA、GAN | 聚类、降维、生成 |
| 半监督学习 | 少量标签 + 大量无标签 | 标签传播、SSL | 数据标注成本高场景 |
| 强化学习 | 延迟奖励，策略优化 | Q-Learning、Policy Gradient | 游戏、机器人、自动驾驶 |

### 2.2 数据集划分原则


In [ ]:
from sklearn.model_selection import train_test_split

# 经典划分：60% 训练 / 20% 验证 / 20% 测试
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# 大数据场景（100万样本）：98% / 1% / 1%
# 小数据场景（1000样本）：70% / 15% / 15% 或 5折交叉验证

# 避免数据泄露：分层抽样保持类别比例
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)


### 2.3 偏差-方差分解

对于均方误差损失，数据集 D 上的期望预测误差：

**E[(y - ŷ)²] = Bias² + Variance + Irreducible Noise**

- **偏差（Bias）**: 模型假设空间与真实函数的距离。高偏差 → 欠拟合。
- **方差（Variance）**: 模型对训练数据的敏感度。高方差 → 过拟合。
- **噪声（Irreducible）**: 数据的固有不确定性，无法消除。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
n_samples = 100
X = np.sort(np.random.uniform(0, 1, n_samples))
y = np.sin(2 * np.pi * X) + np.random.normal(0, 0.3, n_samples)

# 多项式拟合不同复杂度
degrees = [1, 3, 10, 30]
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for idx, deg in enumerate(axes.flat):
    from sklearn.preprocessing import PolynomialFeatures
    from sklearn.linear_model import Ridge
    
    poly = PolynomialFeatures(degree=deg)
    X_poly = poly.fit_transform(X.reshape(-1, 1))
    
    model = Ridge(alpha=0.1)
    model.fit(X_poly, y)
    y_pred = model.predict(X_poly)
    
    axes[idx].scatter(X, y, alpha=0.5, s=10)
    axes[idx].plot(X, y_pred, color='red', linewidth=2)
    axes[idx].set_title(f"degree={deg}")
    axes[idx].set_ylim(-2, 2)

plt.suptitle("偏差-方差权衡：多项式阶数的影响", fontsize=14)
plt.tight_layout()
plt.savefig("l5_bias_variance.png", dpi=150)
plt.show()


### 2.4 过拟合与正则化

**过拟合原因**: 模型复杂度过高，记忆了训练数据的噪声。

**缓解策略**:
1. **L1/L2正则化**: 在损失函数中添加参数惩罚项
   - L2 (Ridge): λ||w||₂² → 参数趋近于零但不全为零
   - L1 (Lasso): λ||w||₁ → 参数稀疏化，可用于特征选择
2. **Dropout** (神经网络): 训练时随机丢弃神经元，强制学习冗余表示
3. **Early Stopping**: 监控验证集性能，在性能开始下降时停止训练
4. **数据增强**: 通过变换扩充训练数据（旋转、平移、噪声等）


In [ ]:
from sklearn.linear_model import Ridge, Lasso

# Ridge回归（L2正则化）
ridge = Ridge(alpha=1.0)  # alpha越大，正则化越强
ridge.fit(X_train, y_train)

# Lasso回归（L1正则化）
lasso = Lasso(alpha=0.1)  # alpha越大，稀疏化越强
lasso.fit(X_train, y_train)
print(f"Lasso非零参数数量: {np.sum(lasso.coef_ != 0)}")


### 2.5 奥卡姆剃刀与VC维

- **奥卡姆剃刀**: 在同样解释力的情况下，优先选择简单的模型
- **VC维 (Vapnik-Chervonenkis Dimension)**: 衡量假设空间复杂度，VC维越高，可拟合的函数种类越多，但过拟合风险越大

## 3. 代码示例


In [ ]:
"""
L5-ml-paradigms.py
演示：四种机器学习范式的核心差异
"""

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification, make_blobs
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans

# ============ 1. 监督学习：分类 ============
print("【监督学习】分类任务")
X_clf, y_clf = make_classification(n_samples=200, n_features=2, n_informative=2,
                                   n_redundant=0, n_clusters_per_class=1, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X_clf, y_clf, test_size=0.3, random_state=42)

model = LogisticRegression()
model.fit(X_train, y_train)
acc = model.score(X_test, y_test)
print(f"逻辑回归分类准确率: {acc:.2%}")

# ============ 2. 无监督学习：聚类 ============
print("\n【无监督学习】聚类任务")
X_blob, y_blob = make_blobs(n_samples=200, centers=3, cluster_std=1.5, random_state=42)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
labels = kmeans.fit_predict(X_blob)
print(f"K-Means聚类完成，轮廓系数: {silhouette_score(X_blob, labels):.3f}")


## 4. 练习题

1. **范式辨析**: 给定四个场景（房价预测、垃圾邮件分类、客户分群、AlphaGo下棋），判断它们分别属于哪种机器学习范式。
2. **数据集划分**: 设计合理的数据划分方案（训练/验证/测试），说明如何保证各类别比例一致。
3. **偏差-方差分析**: 训练误差5%，测试误差35%——分析问题并提出改善措施。
4. **正则化实验**: 比较Ridge和Lasso在不同alpha值下的表现差异。
5. **VC维理解**: 说明线性分类器的VC维及其意义。

## 5. 延伸阅读

- **书籍**: Tom Mitchell, *Machine Learning*（第1版）
- **在线课程**: Andrew Ng, *Machine Learning*（Coursera）
- **工具**: scikit-learn 官方文档 — https://scikit-learn.org/

---
